#### Imports

In [39]:
import pandas as pd
from newspaper import Article, Config
from newspaper.article import ArticleException
from selenium import webdriver
import undetected_chromedriver as uc
from tqdm import tqdm

In [2]:
tqdm.pandas()

#### Setup Selinium Environment

- NDTV and Telegraph require special handling using selinium, as they block traditional scraping methods
- Selinium opens a browser window and fetches a fully loaded HTML file for our processing

In [43]:
options = uc.ChromeOptions()
options.add_argument("--headless=new")
options.add_argument("--window-size=1920,1080")

In [44]:
def get_selenium_html(url):
    with uc.Chrome(options=options) as driver: 
        driver.get(url)
        article_html = driver.page_source
    return article_html

#### Setup Newspaper3k Environment

In [45]:
config = Config()
config.browser_user_agent = 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_6) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124  Safari/537.36'
config.request_timeout = 20

In [46]:
def extract_items(url) -> tuple:
    try:
        article = Article(url, config=config)

        article.download()
        article.parse()

        Text = " ".join(article.text.split()[:200])


        if Text == '':
            raise LookupError(f"Unable to extact article text for {url}, likely blocked.") # if it still fails, can't circumvent.


        parsed_title = article.title
        parsed_authors = ", ".join(article.authors)
        

        #parsed_tags = ", ".join(article.tags)

        article.nlp()

        parsed_summary = article.summary
        parsed_keywords = ", ".join(article.keywords)

        return (parsed_title, parsed_authors, Text, parsed_summary, parsed_keywords)
    
    except (ArticleException, LookupError) as e:
        if str(e).find('403') != -1 or isinstance(e, LookupError):

            article = Article(url, config=config)
            article.download(input_html = get_selenium_html(url))

            article.parse()

            Text = " ".join(article.text.split()[:200])
            if Text == '':
                print(f"Unable to extact article text for {url}, likely blocked.")


            parsed_title = article.title
            parsed_authors = ", ".join(article.authors)
            

            #parsed_tags = ", ".join(article.tags)

            article.nlp()

            parsed_summary = article.summary
            parsed_keywords = ", ".join(article.keywords)

            return (parsed_title, parsed_authors, Text, parsed_summary, parsed_keywords)

        else:
            print(e)
            return (None, None, None, None, None, None,)

In [47]:
extract_items('https://www.ndtv.com/india-news/bjp-wont-allow-bengal-polls-if-sir-process-obstructed-state-party-chief-10745212')

could not detect version_main.therefore, we are assuming it is chrome 108 or higher


Unable to extact article text for https://www.ndtv.com/india-news/bjp-wont-allow-bengal-polls-if-sir-process-obstructed-state-party-chief-10745212, likely blocked.


('Access Denied', '', '', '', 'access, denied')

#### Load CSV

In [18]:
event = 'elections'

In [19]:
df = pd.read_csv(f'{event}_opinions.csv')

df

,id,indexed_date,language,media_name,media_url,publish_date,title,url
0,040f2d7778040bedacec572f66446e31c51e5cc6ab2fb3...,2025-07-02 15:51:38.113666+00:00,en,indianexpress.com,indianexpress.com,2019-06-21,One man show,https://indianexpress.com/article/opinion/colu...
1,8c9b2e13a6d8804482a4834e25af9b51c60218e06920d7...,2025-07-02 15:35:31.872823+00:00,en,dnaindia.com,dnaindia.com,2019-06-21,BJP’s audacious target,https://www.dnaindia.com/analysis/column-bjp-s...
2,3ef9d8469c5dc41b1b0ae8875c065132315a5b0c4a4ffe...,2025-07-02 10:24:58.756950+00:00,en,indiatimes.com,indiatimes.com,2019-06-25,"Yesterday, once more: Mayawati faces tough cha...",https://timesofindia.indiatimes.com/blogs/toi-...
3,dcd2a88950e4f78a478eaf612e83fd71e8015e2c2ab43d...,2025-07-02 07:08:09.645774+00:00,en,indiatoday.in,indiatoday.in,2019-06-27,"With Narendra Modi taking lead, how BJP is app...",https://www.indiatoday.in/news-analysis/story/...
4,208f5885d54138a55c2671fed309e2e038c34314885170...,2025-07-02 05:56:16.345701+00:00,en,thehindu.com,thehindu.com,2019-06-28,"Will the idea of ‘one nation, one poll’ work i...",https://www.thehindu.com/opinion/op-ed/will-th...
...,...,...,...,...,...,...,...,...
1507,ce0653dfb6ca458d65e587e462ae36e9672f765b21eced...,2023-11-21 00:39:23+00:00,en,indianexpress.com,indianexpress.com,2023-11-19,"P Chidambaram writes: Different states, Differ...",https://indianexpress.com/article/opinion/colu...
1508,cdb5a12056dfb4573b7a95e51a6003dc1803506d306512...,2023-11-20 03:26:07+00:00,en,thequint.com,thequint.com,2023-11-18,Telangana Election: What is Congress' Strategy...,https://www.thequint.com/opinion/telangana-ele...
1509,eaabb4162e532903144d582d15b8a8c84a98bcf093fac8...,2023-11-19 02:43:21+00:00,en,hindustantimes.com,hindustantimes.com,2023-11-17,Can INDIA afford to campaign without unifying ...,https://www.hindustantimes.com/opinion/keeping...
1510,fbc8fc2ce95cde01dfcb9e1002b12e751f033f1341a8ee...,2023-11-18 01:03:34+00:00,en,hindustantimes.com,hindustantimes.com,2023-11-16,The battle for central India,https://www.hindustantimes.com/editorials/the-...


#### Extract Text and Various Fields

In [20]:
newspaper3k_df = df['url'].progress_apply(extract_items)

new_column_names = [
    'parsed_title', 'parsed_authors', 
    'Text', 'parsed_tags', 'parsed_summary', 'parsed_keywords'
]

newspaper3k_df = pd.DataFrame(data = newspaper3k_df.tolist(), columns = new_column_names, index=df.index)

newspaper3k_df.head()

 22%|██▏       | 330/1512 [12:15<15:40,  1.26it/s]  

Article `download()` failed with 404 Client Error: Not Found for url: https://www.hindustantimes.com/columns/the-battle-for-the-crown-in-delhi/story-ytmqUUJ3C7dJmQSEKr6aZN.htmlhttps://www.hindustantimes.com/columns/the-battle-for-the-crown-in-delhi/story-ytmqUUJ3C7dJmQSEKr6aZN.html on URL https://www.hindustantimes.com/columns/the-battle-for-the-crown-in-delhi/story-ytmqUUJ3C7dJmQSEKr6aZN.htmlhttps://www.hindustantimes.com/columns/the-battle-for-the-crown-in-delhi/story-ytmqUUJ3C7dJmQSEKr6aZN.html


 23%|██▎       | 346/1512 [12:29<14:54,  1.30it/s]

Article `download()` failed with 404 Client Error: Not Found for url: http://rss.cnn.com/~r/rss/cnn_topstories/~3/d5FdLBEOo0c/index.html on URL http://rss.cnn.com/~r/rss/cnn_topstories/~3/d5FdLBEOo0c/index.html
Article `download()` failed with 404 Client Error: Not Found for url: http://rss.cnn.com/~r/rss/cnn_latest/~3/d5FdLBEOo0c/index.html on URL http://rss.cnn.com/~r/rss/cnn_latest/~3/d5FdLBEOo0c/index.html


 54%|█████▎    | 809/1512 [30:10<1:28:26,  7.55s/it]

Unable to extact article text for https://economictimes.indiatimes.com/news/newsblogs/exit-poll-2024-results-live-updates-india-lok-sabha-elections-opinion-polls-c-voter-survey-chanakya-axis-my-india-bjp-congress-aap-tmc-1st-june-today-news/liveblog/110607664.cms, likely blocked.


 91%|█████████ | 1369/1512 [52:55<03:39,  1.53s/it]  

Article `download()` failed with 404 Client Error: Not Found for url: https://www.dnaindia.com/analysis/report-women-s-reservation-bill-women-voters-mps-from-all-parties-voted-aye-for-you-3062877 on URL https://www.dnaindia.com/analysis/report-women-s-reservation-bill-women-voters-mps-from-all-parties-voted-aye-for-you-3062877


100%|██████████| 1512/1512 [57:17<00:00,  2.27s/it]


,parsed_title,parsed_authors,Text,parsed_tags,parsed_summary,parsed_keywords
0,One man show,"Christophe Jaffrelot, Cdata, Var Template_Cont...",The just-concluded Lok Sabha elections were wo...,,But the Congress won comfortably in most of th...,"cent, won, state, states, modi, party, congres..."
1,BJP’s audacious target,Amitabh Tiwari,Party has set an ambitious goal of dislodging ...,"opinion, Prashant Kishore, Bharatiya Janata Pa...",Critics who were giving BJP not more than 8-10...,"vote, bjps, audacious, tmc, lok, state, sabha,..."
2,"Yesterday, once more: Mayawati faces tough cha...",,Bahujan Samaj Party chief Mayawati yesterday a...,,Bahujan Samaj Party chief Mayawati yesterday a...,"playbook, yesterday, faces, wont, resorting, a..."
3,"With Narendra Modi taking lead, how BJP is app...",,"Nationalism, protection of cows from slaughter...",,"Incidentally, in 2019 Lok Sabha election, PM M...","lead, gandhi, taking, appropriating, narendra,..."
4,"Parley | Will the idea of ‘one nation, one pol...",Authors,"Last week, when Prime Minister Narendra Modi c...","Data Point Podcast, ISRO, Ground Zero, Spotlig...",A State election should be fought by the peopl...,"parties, parley, simultaneous, money, idea, in..."


In [21]:
df = df.join(newspaper3k_df)

df.head()

,id,indexed_date,language,media_name,media_url,publish_date,title,url,parsed_title,parsed_authors,Text,parsed_tags,parsed_summary,parsed_keywords
0,040f2d7778040bedacec572f66446e31c51e5cc6ab2fb3...,2025-07-02 15:51:38.113666+00:00,en,indianexpress.com,indianexpress.com,2019-06-21,One man show,https://indianexpress.com/article/opinion/colu...,One man show,"Christophe Jaffrelot, Cdata, Var Template_Cont...",The just-concluded Lok Sabha elections were wo...,,But the Congress won comfortably in most of th...,"cent, won, state, states, modi, party, congres..."
1,8c9b2e13a6d8804482a4834e25af9b51c60218e06920d7...,2025-07-02 15:35:31.872823+00:00,en,dnaindia.com,dnaindia.com,2019-06-21,BJP’s audacious target,https://www.dnaindia.com/analysis/column-bjp-s...,BJP’s audacious target,Amitabh Tiwari,Party has set an ambitious goal of dislodging ...,"opinion, Prashant Kishore, Bharatiya Janata Pa...",Critics who were giving BJP not more than 8-10...,"vote, bjps, audacious, tmc, lok, state, sabha,..."
2,3ef9d8469c5dc41b1b0ae8875c065132315a5b0c4a4ffe...,2025-07-02 10:24:58.756950+00:00,en,indiatimes.com,indiatimes.com,2019-06-25,"Yesterday, once more: Mayawati faces tough cha...",https://timesofindia.indiatimes.com/blogs/toi-...,"Yesterday, once more: Mayawati faces tough cha...",,Bahujan Samaj Party chief Mayawati yesterday a...,,Bahujan Samaj Party chief Mayawati yesterday a...,"playbook, yesterday, faces, wont, resorting, a..."
3,dcd2a88950e4f78a478eaf612e83fd71e8015e2c2ab43d...,2025-07-02 07:08:09.645774+00:00,en,indiatoday.in,indiatoday.in,2019-06-27,"With Narendra Modi taking lead, how BJP is app...",https://www.indiatoday.in/news-analysis/story/...,"With Narendra Modi taking lead, how BJP is app...",,"Nationalism, protection of cows from slaughter...",,"Incidentally, in 2019 Lok Sabha election, PM M...","lead, gandhi, taking, appropriating, narendra,..."
4,208f5885d54138a55c2671fed309e2e038c34314885170...,2025-07-02 05:56:16.345701+00:00,en,thehindu.com,thehindu.com,2019-06-28,"Will the idea of ‘one nation, one poll’ work i...",https://www.thehindu.com/opinion/op-ed/will-th...,"Parley | Will the idea of ‘one nation, one pol...",Authors,"Last week, when Prime Minister Narendra Modi c...","Data Point Podcast, ISRO, Ground Zero, Spotlig...",A State election should be fought by the peopl...,"parties, parley, simultaneous, money, idea, in..."


In [22]:
df.to_csv(f'Extracted/{event}_opinions.csv', index=False)